# 04 — Preprocessing and Modelling

**Dataset:** Global Coral Bleaching Database 1980–2020 (van Woesik & Kratochwill 2022)
**Target variable:** `Bleaching_Category`

---

## Contents

1. Preprocessing
   - 1.1 Define feature types
   - 1.2 Exploratory check on categorical columns and encoding strategy
   - 1.3 Temporal train/validation/test split
   - 1.4 Preprocessing pipeline
2. Modelling — 4-class baseline (none / mild / moderate / severe)
   - 2.1 Logistic Regression
   - 2.2 Random Forest
   - 2.3 XGBoost
   - 2.4 Baseline model comparison
3. Addressing class imbalance
4. TimeSeriesSplit cross-validation (abandoned)
5. Binary classification + SMOTE (none / bleaching)
   - 5.1 Random Forest vs XGBoost comparison
6. Error analysis
   - 6.1 False negative analysis
   - 6.1.1 False positive analysis
   - 6.2 Feature ablation: removing Date_Year
   - 6.2.1 Date_Year ablation results
7. Hyperparameter tuning: GridSearchCV (abandoned) and manual tuning
   - 7.1 GridSearchCV v1 results
   - 7.2 GridSearchCV v2 results
   - 7.3 Manual tuning results
8. Model Interpretability: SHAP
   - 8.1 SHAP results and interpretation
9. Regional Model — Atlantic Ocean
   - 9.1 Why a regional Atlantic model makes sense

---

## Key methodological decisions

- **Temporal split:** train on 1980–2009, validate on 2010–2014, evaluate on 2015–2019.
  The three major bleaching events (1998, 2005, 2016) are distributed across the three sets.
- **Class imbalance:** handled via `class_weight='balanced'` in baseline models (§2).
  In the binary classification model (§5), SMOTE is used instead — combining both
  mechanisms is redundant as they address the same problem.
- **`Date_Year` as feature:** retained as a conscious choice. The temporal trend 
  (more bleaching in recent years) is ecologically real and directly supported by 
  thermal variables in the dataset. Ablation in §6.2 confirms no measurable 
  temporal overfitting.
- **Primary metric:** recall — missing a bleaching event carries higher ecological cost
  than a false alarm.
- **Secondary metric:** F1-score.
- **Encoding:** one-hot encoding for all categorical features (max 8 unique values per column).
- **Scaling:** StandardScaler for all numerical features, fitted on train only.

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV
from sklearn.compose import ColumnTransformer
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
import shap

# Setting pandas to show all columns: the dataset has 27 columns and pandas truncates by default
pd.set_option('display.max_columns', None)

In [2]:
# Project colour palette: ocean themed 🌊
color_primary = 'teal'
color_secondary = 'coral'  
color_accent = 'turquoise'

In [3]:
# Importing the data post-feature selection
df = pd.read_csv('../data/processed/bleaching_feature_selection.csv', low_memory=False)
print(df.shape)
df.head()

(32711, 27)


,Latitude_Degrees,Longitude_Degrees,Ocean_Name,Realm_Name,Distance_to_Shore,Exposure,Turbidity,Cyclone_Frequency,Date_Month,Date_Year,Depth_m,Substrate_Name,ClimSST,Temperature_Kelvin,Temperature_Mean,Temperature_Maximum,Windspeed,SSTA,SSTA_DHW,TSA,TSA_Maximum,TSA_Mean,TSA_Frequency,TSA_DHW,TSA_DHWMax,TSA_DHWMean,Bleaching_Category
0,23.163,-82.5260,Atlantic,Tropical Atlantic,8519.23,Exposed,0.0287,49.90,9,2005,10.00,Not_Recorded,301.61,302.05,300.67,304.69,8.0,-0.46,0.00,-0.80,1.83,-2.17,0.00,0.00,7.25,0.18,moderate
1,-17.575,-149.7833,Pacific,Eastern Indo-Pacific,1431.62,Exposed,0.0262,51.20,3,1991,14.00,Not_Recorded,262.15,303.30,300.73,305.01,2.0,1.29,0.26,1.29,3.00,-1.26,0.25,0.26,4.65,0.19,moderate
2,18.369,-64.5640,Atlantic,Tropical Atlantic,182.33,Exposed,0.0429,61.52,1,2006,7.00,Not_Recorded,298.79,299.18,300.32,304.14,8.0,0.04,0.00,-2.64,2.31,-1.49,7.00,0.00,11.66,0.26,moderate
3,17.760,-64.5680,Atlantic,Tropical Atlantic,313.13,Exposed,0.0424,65.39,4,2006,9.02,Not_Recorded,300.16,299.61,300.38,304.07,3.0,-0.07,0.00,-2.27,2.19,-1.49,3.00,0.00,5.64,0.20,moderate
4,17.769,-64.5830,Atlantic,Tropical Atlantic,792.00,Exposed,0.0424,65.39,4,2006,12.50,Not_Recorded,300.15,299.70,300.38,303.76,3.0,0.00,0.00,-2.19,1.87,-1.50,3.00,0.00,6.89,0.25,moderate


In [4]:
# Sanity check: no missing values expected after feature selection
df.isna().sum()

Latitude_Degrees       0
Longitude_Degrees      0
Ocean_Name             0
Realm_Name             0
Distance_to_Shore      0
Exposure               0
Turbidity              0
Cyclone_Frequency      0
Date_Month             0
Date_Year              0
Depth_m                0
Substrate_Name         0
ClimSST                0
Temperature_Kelvin     0
Temperature_Mean       0
Temperature_Maximum    0
Windspeed              0
SSTA                   0
SSTA_DHW               0
TSA                    0
TSA_Maximum            0
TSA_Mean               0
TSA_Frequency          0
TSA_DHW                0
TSA_DHWMax             0
TSA_DHWMean            0
Bleaching_Category     0
dtype: int64

## 1. Preprocessing

### 1.1 Define feature types

We separate the 26 features into two groups:
- **Numerical**: continuous variables that need scaling
- **Categorical**: string variables that need encoding

This separation is needed to apply different preprocessing steps to each group.

In [5]:
num_cols = [
    'Latitude_Degrees', 'Longitude_Degrees', 'Distance_to_Shore',
    'Turbidity', 'Cyclone_Frequency', 'Date_Month', 'Date_Year',
    'Depth_m', 'ClimSST', 'Temperature_Kelvin', 'Temperature_Mean',
    'Temperature_Maximum', 'Windspeed', 'SSTA', 'SSTA_DHW',
    'TSA', 'TSA_Maximum', 'TSA_Mean', 'TSA_Frequency',
    'TSA_DHW', 'TSA_DHWMax', 'TSA_DHWMean']

cat_cols = ['Ocean_Name', 'Realm_Name', 'Substrate_Name', 'Exposure']

target = 'Bleaching_Category'

### 1.2 Exploratory check on categorical string columns

Before building the preprocessing pipeline, we inspect the categorical string columns
to understand how many unique values each one contains. This informs our encoding choices.

In [6]:
# Inspection of categorical columns
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- Ocean_Name ---
Ocean_Name
Pacific         16264
Atlantic        13087
Indian           1982
Red Sea          1037
Arabian Gulf      341
Name: count, dtype: int64

--- Realm_Name ---
Realm_Name
Central Indo-Pacific           13793
Tropical Atlantic              13076
Western Indo-Pacific            3140
Eastern Indo-Pacific            1618
Temperate Australasia            532
Temperate Northern Pacific       491
Tropical Eastern Pacific          50
Temperate Northern Atlantic       11
Name: count, dtype: int64

--- Substrate_Name ---
Substrate_Name
Hard Coral                  11178
Nutrient Indicator Algae    10960
Not_Recorded                10355
Fleshy Seaweed                218
Name: count, dtype: int64

--- Exposure ---
Exposure
Sheltered    18240
Exposed      11794
Sometimes     2677
Name: count, dtype: int64


### Encoding strategy for categorical columns

All four categorical columns have low cardinality (between 3 and 8 unique values).
One-hot encoding is a viable strategy for all of them. Using `drop='first'` to avoid 
multicollinearity, it will add 16 new columns in total (4 + 7 + 3 + 2), keeping the 
feature space manageable.

### 1.3 Train/Validation/Test split

In [7]:
# Checking the number of observation per year before deciding on the best test/validation/train strategy
print(df.groupby('Date_Year').size())

Date_Year
1983       4
1987      16
1988       1
1990       1
1991       4
1992       5
1993       6
1994      11
1995       6
1996      10
1997       9
1998     177
1999     721
2000     549
2001     703
2002     565
2003    1644
2004    1677
2005    3886
2006    2775
2007    1519
2008    1901
2009    1931
2010    1549
2011    1356
2012    1400
2013    1517
2014    1409
2015    1558
2016    1803
2017    1581
2018    1243
2019    1174
dtype: int64


In [8]:
# Inspecting the number of observations for the proposed train/val/test split
periods = {
    '1980-2009': (1980, 2009),
    '2010-2014': (2010, 2014),
    '2015-2019': (2015, 2019)
}

for name, (start, end) in periods.items():
    count = df[(df['Date_Year'] >= start) & (df['Date_Year'] <= end)].shape[0]
    print(f"{name}: {count} observations")

1980-2009: 18121 observations
2010-2014: 7231 observations
2015-2019: 7359 observations


### Temporal Train/Validation/Test split

We split the dataset into three periods based on year:

- **Train:      1980–2009  →  18,121 observations**
- **Validation: 2010–2014  →   7,231 observations**
- **Test:       2015–2019  →   7,359 observations**

This split is chosen because it respects the temporal structure of the data
and ensures that the three major documented bleaching events are distributed meaningfully:
the 1998 and 2005 events fall in the training set, the 2010 event in validation,
and the 2016 event in the test set, which the model will only see at final evaluation.

The scarce pre-1997 data is not a concern: systematic coral bleaching monitoring
only became widespread after the 1997–1998 El Niño event, so the low observation
count before that date reflects the historical record, not a data quality issue.

In [9]:
# Splitting the dataset into training, validation and testing subsets
train = df[df['Date_Year'] <= 2009]
validation = df[(df['Date_Year'] >= 2010) & (df['Date_Year'] <= 2014)]
test = df[df['Date_Year'] >= 2015]

X_train = train.drop(columns='Bleaching_Category')
y_train = train['Bleaching_Category']

X_val = validation.drop(columns='Bleaching_Category')
y_val = validation['Bleaching_Category']

X_test = test.drop(columns='Bleaching_Category')
y_test = test['Bleaching_Category']

print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

Train:      (18121, 26)
Validation: (7231, 26)
Test:       (7359, 26)


### 1.4 Preprocessing pipeline

We build a preprocessing pipeline using `ColumnTransformer` from sklearn.
This applies two transformations in parallel:
- `StandardScaler` on numerical columns
- `OneHotEncoder` on categorical columns

The pipeline is fitted on the training set only, then applied to validation and test.
This prevents data leakage from future observations into the training process.

In [10]:
# Preprocessing pipeline: StandardScaler for numerical columns, OneHotEncoder for categorical columns
# The pipeline is fitted on the training set only to prevent data leakage
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

In [11]:
# Fit on train only, then transform all three sets
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"Train processed:      {X_train_processed.shape}")
print(f"Validation processed: {X_val_processed.shape}")
print(f"Test processed:       {X_test_processed.shape}")

Train processed:      (18121, 38)
Validation processed: (7231, 38)
Test processed:       (7359, 38)


### A note on fit_transform() Vs transform()

- `fit_transform(X_train)`: learns the statistics from the training set (e.g. mean and 
  standard deviation for scaling, unique categories for encoding) and applies the 
  transformation. This is called only once, on the training set.

- `transform(X_val)` and `transform(X_test)`: applies the same transformation learned 
  from the training set, without relearning anything. This ensures that validation and 
  test are scaled and encoded using training statistics only, preventing data leakage.

## 2. Modelling

### 2.1 Logistic Regression baseline model

Logistic Regression is our simplest model and serves as a baseline.
We use `class_weight='balanced'` to handle class imbalance, and evaluate
on the validation set using recall as primary metric and F1 as secondary.

Logistic Regression has regularisation built in by default (C=1.0), 
so no additional constraints are needed for the baseline.

In [12]:
# Train the model
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_processed, y_train)

# Predict on validation set
y_pred_lr = lr.predict(X_val_processed)

# Evaluation
print(classification_report(y_val, y_pred_lr))

              precision    recall  f1-score   support

        mild       0.19      0.59      0.29       533
    moderate       0.15      0.25      0.19       255
        none       0.96      0.73      0.83      6327
      severe       0.12      0.34      0.18       116

    accuracy                           0.70      7231
   macro avg       0.36      0.48      0.37      7231
weighted avg       0.86      0.70      0.76      7231



In [ ]:
cm = confusion_matrix(y_val, y_pred_lr, labels=['none', 'mild', 'moderate', 'severe'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'mild', 'moderate', 'severe'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('Logistic Regression — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

### 2.1.1 Logistic Regression — Results

**Classification report highlights:**
- `none` is the only well-performing class: recall 0.73, F1 0.83.
- `severe` recall is 0.34. The model misses 2 out of 3 severe bleaching events.
  This is the most ecologically costly error in this project.
- `moderate` recall is 0.25. More than half of moderate bleaching cases are 
  misclassified as mild or none.
- Overall accuracy of 70% is misleading: it is inflated by the dominance of the 
  `none` class (6,327 out of 7,231 validation observations).
- **Macro F1: 0.37**. This is the honest summary metric, as it treats all classes equally.

**Confusion matrix highlights:**
- Of 116 severe cases, only 40 are correctly identified. 39 are predicted as mild, 
  26 as moderate: the model systematically underestimates bleaching severity.
- Of 255 moderate cases, only 63 are correctly identified.

**Conclusion:** Logistic Regression captures the dominant class well but fails on minority 
classes. This is expected for a linear baseline — it establishes the performance floor 
and motivates the use of more expressive models.

### 2.2 Random Forest baseline model

Random Forest builds an ensemble of decision trees, each trained on a random subset 
of the data and features. It captures non-linear relationships that Logistic Regression 
cannot.

To avoid pathological overfitting from fully grown trees, we set `max_depth=10` and 
`min_samples_leaf=5` as minimum constraints. `class_weight='balanced'` handles class imbalance.

In [14]:
# Train the model
rf = RandomForestClassifier(
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
rf.fit(X_train_processed, y_train)

# Predict on validation set
y_pred_rf = rf.predict(X_val_processed)

# Evaluation
print(classification_report(y_val, y_pred_rf))

              precision    recall  f1-score   support

        mild       0.28      0.55      0.37       533
    moderate       0.27      0.17      0.21       255
        none       0.94      0.89      0.92      6327
      severe       0.30      0.15      0.20       116

    accuracy                           0.83      7231
   macro avg       0.45      0.44      0.42      7231
weighted avg       0.86      0.83      0.84      7231



In [ ]:
cm = confusion_matrix(y_val, y_pred_rf, labels=['none', 'mild', 'moderate', 'severe'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'mild', 'moderate', 'severe'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('Random Forest — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

### 2.2.1 Random Forest — Results

**Classification report highlights:**
- Macro F1 improves from 0.37 (Logistic Regression) to 0.42 — a real but modest gain.
- `none` improves significantly: recall 0.73 → 0.89.
- `mild` improves slightly: F1 0.29 → 0.37. The model correctly identifies 294/533 
  mild cases, but 222 are misclassified as none.
- `moderate` worsens: recall 0.25 → 0.17. 75 moderate cases are predicted as none.
- `severe` worsens significantly: recall 0.34 → 0.15. Only 17 out of 116 severe 
  bleaching events are correctly identified. 39 are predicted as none.

**Confusion matrix highlights:**
- The model becomes more conservative on minority classes compared to Logistic Regression,
  trading recall on severe and moderate for better performance on none.
- `class_weight='balanced'` is insufficient to compensate for the extreme imbalance 
  (6,327 none out of 7,231 validation observations).

**Conclusion:** Random Forest captures the dominant class well but systematically 
underestimates bleaching severity. The core problem is the class imbalance.
If XGBoost shows the same pattern, more aggressive rebalancing strategies 
will be needed during tuning (e.g. SMOTE, adjusted class weights, threshold tuning).

### 2.3 XGBoost baseline model

XGBoost builds trees sequentially, where each tree corrects the errors of the previous one.
It has regularisation built in by default, so vanilla parameters are already reasonable.
Class imbalance is handled by passing sample weights computed from the training labels,
since XGBoost multiclass does not support `class_weight='balanced'` natively.

In [ ]:
# Compute sample weights to handle class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Encode target labels as integers (XGBoost requires numeric labels)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

# Train the model
xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train_processed, y_train_enc, sample_weight=sample_weights)

# Predict on validation set
y_pred_xgb_enc = xgb_model.predict(X_val_processed)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

# Evaluation
print(classification_report(y_val, y_pred_xgb))

In [ ]:
cm = confusion_matrix(y_val, y_pred_xgb, labels=['none', 'mild', 'moderate', 'severe'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'mild', 'moderate', 'severe'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('XGBoost — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

### 2.3.1 XGBoost — Results

**Classification report highlights:**
- Macro F1: 0.37 — identical to Logistic Regression, worse than Random Forest (0.42).
- `none`: recall 0.92, F1 0.92 — the best performance on this class across all three models.
- `mild`: recall 0.47, F1 0.36 — only 248/533 correctly identified, 270 predicted as none.
- `moderate`: recall 0.04, F1 0.06 — the worst performance across all three models. 
  116 out of 255 moderate cases predicted as none.
- `severe`: recall 0.09, F1 0.15 — only 11/116 severe bleaching events correctly identified.
  63 predicted as none.

**Confusion matrix highlights:**
- XGBoost is the most conservative model on minority classes: it predicts none 
  more aggressively than both Logistic Regression and Random Forest.
- Sample weights via `compute_sample_weight` are insufficient to compensate 
  for the extreme class imbalance.

**Conclusion:** XGBoost vanilla performs worst on the ecologically critical classes 
(severe and moderate). It will require the most aggressive tuning to become useful.

## 3. Baseline Model Comparison and Class Imbalance

All three baseline models show the same failure pattern: they predict `none` 
aggressively and systematically underestimate bleaching severity.
This is not a model quality problem, it is a class imbalance problem.

**Class distribution in the validation set:**
- none: 6,327 (87.5%)
- mild: 533 (7.4%)
- moderate: 255 (3.5%)
- severe: 116 (1.6%)

| Model               | Accuracy | Macro F1 | Recall (none) | Recall (mild) | Recall (moderate) | Recall (severe) |
|---------------------|----------|----------|---------------|---------------|-------------------|-----------------|
| Logistic Regression | 0.70     | 0.37     | 0.73          | 0.59          | 0.25              | 0.34            |
| Random Forest       | 0.83     | 0.42     | 0.89          | 0.55          | 0.17              | 0.15            |
| XGBoost             | 0.84     | 0.37     | 0.92          | 0.47          | 0.04              | 0.09            |

`class_weight='balanced'` and sample weights are insufficient when the imbalance 
is this extreme. More aggressive strategies are needed.

**Options considered:**
1. Reduce to 3 classes: merge `moderate` and `severe` into `high`.
2. Reduce to binary: `none` vs `bleaching`.
3. Keep 4 classes with aggressive tuning: SMOTE, threshold tuning, adjusted weights.

**Decision:** reduce to binary classification and apply SMOTE. If performance 
remains insufficient, regional models per ecoregion will be explored as future work.

## 4. TimeSeriesSplit Cross-Validation

The fixed validation period (2010–2014) is a relatively quiet period with limited 
bleaching events, making it unrepresentative for model evaluation. We switch to 
TimeSeriesSplit cross-validation on the full pre-2015 data, which evaluates the model 
across multiple time windows including the 1998 and 2005 major bleaching events.

The test set (2015–2019) remains untouched and will only be used for final evaluation.

In [18]:
# Full pre-2015 data for cross-validation
train_ts = df[df['Date_Year'] < 2015]
test_ts = df[df['Date_Year'] >= 2015]

X_train_ts = train_ts.drop(columns='Bleaching_Category')
y_train_ts = train_ts['Bleaching_Category']

X_test_ts = test_ts.drop(columns='Bleaching_Category')
y_test_ts = test_ts['Bleaching_Category']

print(f"Train: {X_train_ts.shape}")
print(f"Test:  {X_test_ts.shape}")

Train: (25352, 26)
Test:  (7359, 26)


In [19]:
# Separate preprocessor for TimeSeriesSplit
# It avoids overwriting the fit from earlier sections if cells are re-executed out of order
preprocessor_ts = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

X_train_ts_processed = preprocessor_ts.fit_transform(X_train_ts)
X_test_ts_processed = preprocessor_ts.transform(X_test_ts)

print(f"Train processed: {X_train_ts_processed.shape}")
print(f"Test processed:  {X_test_ts_processed.shape}")

Train processed: (25352, 38)
Test processed:  (7359, 38)


In [20]:
# TimeSeriesSplit cross-validation on Random Forest
tscv = TimeSeriesSplit(n_splits=5)

rf_ts = RandomForestClassifier(
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

scores = cross_val_score(
    rf_ts, 
    X_train_ts_processed, 
    y_train_ts, 
    cv=tscv, 
    scoring='f1_macro'
)

print(f"F1 macro per fold: {scores.round(3)}")
print(f"Mean F1 macro:     {scores.mean().round(3)}")
print(f"Std:               {scores.std().round(3)}")

F1 macro per fold: [0.206 0.25  0.236 0.246 0.231]
Mean F1 macro:     0.234
Std:               0.015


## 5. Binary Classification with SMOTE

TimeSeriesSplit cross-validation produced worse results than the fixed validation 
split (mean macro F1: 0.234 vs 0.42), likely due to the sparse bleaching data in 
early time windows. We abandon this approach.

Given the extreme class imbalance and the failure of all three baseline models to 
detect minority classes reliably, we adopt two strategies in combination:

1. **Binary target:** `none` vs `bleaching` (merging mild, moderate, severe)
2. **SMOTE** (Synthetic Minority Oversampling Technique): generates synthetic 
   observations of the minority class in the training set to balance the distribution

The test set remains untouched: SMOTE is applied only to the training data.

In [21]:
# Remap target to binary
df['Bleaching_Binary'] = df['Bleaching_Category'].apply(
    lambda x: 'none' if x == 'none' else 'bleaching'
)
print(df['Bleaching_Binary'].value_counts())

Bleaching_Binary
none         26929
bleaching     5782
Name: count, dtype: int64


In [22]:
# Train/Validation/Test split
train_bin = df[df['Date_Year'] <= 2009]
validation_bin = df[(df['Date_Year'] >= 2010) & (df['Date_Year'] <= 2014)]
test_bin = df[df['Date_Year'] >= 2015]

X_train_bin = train_bin.drop(columns=['Bleaching_Category', 'Bleaching_Binary'])
y_train_bin = train_bin['Bleaching_Binary']

X_val_bin = validation_bin.drop(columns=['Bleaching_Category', 'Bleaching_Binary'])
y_val_bin = validation_bin['Bleaching_Binary']

X_test_bin = test_bin.drop(columns=['Bleaching_Category', 'Bleaching_Binary'])
y_test_bin = test_bin['Bleaching_Binary']

print(f"Train:      {X_train_bin.shape}")
print(f"Validation: {X_val_bin.shape}")
print(f"Test:       {X_test_bin.shape}")

Train:      (18121, 26)
Validation: (7231, 26)
Test:       (7359, 26)


In [23]:
# Separate preprocessor for binary classification.
# It avoids overwriting the fit from earlier sections if cells are re-executed out of order
preprocessor_bin = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

X_train_bin_processed = preprocessor_bin.fit_transform(X_train_bin)
X_val_bin_processed = preprocessor_bin.transform(X_val_bin)
X_test_bin_processed = preprocessor_bin.transform(X_test_bin)

print(f"Train processed:      {X_train_bin_processed.shape}")
print(f"Validation processed: {X_val_bin_processed.shape}")
print(f"Test processed:       {X_test_bin_processed.shape}")

Train processed:      (18121, 38)
Validation processed: (7231, 38)
Test processed:       (7359, 38)


In [24]:
# Check class distribution before SMOTE
print("Before SMOTE:")
print(y_train_bin.value_counts())

Before SMOTE:
Bleaching_Binary
none         14359
bleaching     3762
Name: count, dtype: int64


In [25]:
# SMOTE generates synthetic observations of the minority class (bleaching) until both classes are balanced.
# Applied only to the training set.
# Validation and test remain untouched to reflect real-world class distribution.
smote = SMOTE(random_state=42)
X_train_bin_smote, y_train_bin_smote = smote.fit_resample(X_train_bin_processed, y_train_bin)

print("After SMOTE:")
print(y_train_bin_smote.value_counts())
print(f"\nShape: {X_train_bin_smote.shape}")

After SMOTE:
Bleaching_Binary
bleaching    14359
none         14359
Name: count, dtype: int64

Shape: (28718, 38)


In [26]:
# Applying Random Forest to the balanced dataset.
rf_bin = RandomForestClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
rf_bin.fit(X_train_bin_smote, y_train_bin_smote)

y_pred_rf_bin = rf_bin.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_rf_bin))

              precision    recall  f1-score   support

   bleaching       0.47      0.61      0.53       904
        none       0.94      0.90      0.92      6327

    accuracy                           0.86      7231
   macro avg       0.71      0.76      0.73      7231
weighted avg       0.88      0.86      0.87      7231



In [ ]:
cm = confusion_matrix(y_val_bin, y_pred_rf_bin, labels=['none', 'bleaching'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'bleaching'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('Random Forest Binary + SMOTE — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

In [28]:
# Separate LabelEncoder for binary XGBoost.
# It  avoids overwriting le from 2.3.
le_bin = LabelEncoder()
y_train_bin_smote_enc = le_bin.fit_transform(y_train_bin_smote)
y_val_bin_enc = le_bin.transform(y_val_bin)

xgb_bin = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_bin.fit(X_train_bin_smote, y_train_bin_smote_enc)

y_pred_xgb_bin_enc = xgb_bin.predict(X_val_bin_processed)
y_pred_xgb_bin = le_bin.inverse_transform(y_pred_xgb_bin_enc)

print(classification_report(y_val_bin, y_pred_xgb_bin))

              precision    recall  f1-score   support

   bleaching       0.49      0.41      0.45       904
        none       0.92      0.94      0.93      6327

    accuracy                           0.87      7231
   macro avg       0.70      0.67      0.69      7231
weighted avg       0.86      0.87      0.87      7231



In [ ]:
cm = confusion_matrix(y_val_bin, y_pred_xgb_bin, labels=['none', 'bleaching'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'bleaching'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('XGBoost Binary + SMOTE — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

## 5.1 Binary Classification + SMOTE — Model Comparison

| Model         | Accuracy | Macro F1 | Recall (bleaching) | Recall (none) |
|---------------|----------|----------|--------------------|---------------|
| Random Forest | 0.86     | 0.73     | 0.61               | 0.90          |
| XGBoost       | 0.87     | 0.69     | 0.41               | 0.94          |

**Random Forest is the better model for this project.**

- RF recall on `bleaching`: 0.61. Correctly identifies 555 out of 904 bleaching events.
  349 bleaching events are missed (false negatives).
- XGBoost recall on `bleaching`: 0.41. Misses 533 out of 904 bleaching events.
  Higher accuracy but worse on the metric that matters ecologically.
- XGBoost trades bleaching recall for better none recall (0.94 vs 0.90). 
  The wrong trade-off for this problem.

**SMOTE alone is sufficient for RF** — recall 0.61 is identical to the earlier version
that combined SMOTE with `class_weight='balanced'`. The two mechanisms are redundant:
SMOTE already balances the training set, so `class_weight` adds nothing on top.
SMOTE provided a meaningful improvement for XGBoost but performance remains 
below RF on the primary metric.

**Conclusion:** Random Forest Binary + SMOTE is selected as the model to tune.
Target: improve bleaching recall above 0.61 while keeping precision above 0.40.
Next step: GridSearchCV on `max_depth`, `n_estimators`, `min_samples_leaf`.

## 6. Error Analysis

Before tuning, we analyse where the model fails systematically.
Understanding which observations are misclassified (by year, ocean, and realm)
informs tuning decisions and helps identify structural limitations of the global model.

In [30]:
# Add predictions to validation set for error analysis
val_analysis = X_val_bin.copy()
val_analysis['y_true'] = y_val_bin.values
val_analysis['y_pred'] = y_pred_rf_bin

# False negatives: bleaching events predicted as none
false_negatives = val_analysis[
    (val_analysis['y_true'] == 'bleaching') & 
    (val_analysis['y_pred'] == 'none')
]

print(f"Total false negatives: {len(false_negatives)}")
print(f"Total bleaching events: {len(val_analysis[val_analysis['y_true'] == 'bleaching'])}")
print(f"Miss rate: {len(false_negatives) / len(val_analysis[val_analysis['y_true'] == 'bleaching']):.2f}")

Total false negatives: 349
Total bleaching events: 904
Miss rate: 0.39


In [31]:
# False negatives by ocean
print("--- False negatives by Ocean ---")
print(false_negatives['Ocean_Name'].value_counts())

print("\n--- False negatives by Realm ---")
print(false_negatives['Realm_Name'].value_counts())

print("\n--- False negatives by Year ---")
print(false_negatives['Date_Year'].value_counts().sort_index())

--- False negatives by Ocean ---
Ocean_Name
Pacific         236
Atlantic        107
Red Sea           4
Arabian Gulf      2
Name: count, dtype: int64

--- False negatives by Realm ---
Realm_Name
Central Indo-Pacific          204
Tropical Atlantic             107
Temperate Northern Pacific     12
Temperate Australasia          12
Western Indo-Pacific            6
Tropical Eastern Pacific        6
Eastern Indo-Pacific            2
Name: count, dtype: int64

--- False negatives by Year ---
Date_Year
2010    139
2011     40
2012     24
2013     70
2014     76
Name: count, dtype: int64


### 6.1 False Negative Analysis

The model misses 349 out of 904 bleaching events (miss rate: 0.39).

**By Ocean:**
- Pacific: 236 false negatives (67% of all false negatives)
- Atlantic: 107 false negatives (31%)
- Red Sea and Arabian Gulf: 6 combined

**By Realm:**
- Central Indo-Pacific: 204 false negatives (58% of all false negatives):
  disproportionate given it represents ~42% of total observations. 
  The model has a systematic geographic bias against this region.
- Tropical Atlantic: 107 false negatives: all Atlantic errors fall here,
  confirming that Ocean_Name and Realm_Name are partially redundant.

**By Year:**
- 2010 accounts for 139 out of 349 false negatives (40% of all errors in a single year).
  The 2010 event is a moderate post-El Niño bleaching episode: its thermal signature 
  may differ from the major 1998 and 2005 events the model learned from.
- 2011–2014 are relatively homogeneous (24–76 false negatives per year).

**Key findings:**
- The model has a systematic bias against the Central Indo-Pacific region.
- Ocean_Name and Realm_Name contain overlapping information: 
  one of the two may be redundant as a feature.
- The 2010 anomaly warrants further investigation during tuning.

**Structural limitations:**
- The training set (1980–2009) is dominated by Atlantic and Central Indo-Pacific 
  observations. Temperate regions and the Eastern Pacific are severely underrepresented, 
  which explains the near-zero recall in those areas.
- A dataset extending to 2024 or beyond would significantly improve model performance 
  for these regions: the Great Barrier Reef and Indo-Pacific have experienced major 
  bleaching events in 2016, 2017, 2020, 2022, and 2024. Including those observations 
  would give the model the signal it currently lacks for temperate and Pacific realms.
- Regional models are recommended as future work, after the current global model is finalised.

In [32]:
# Compare false negative rate by ocean vs total bleaching events
total_bleaching = val_analysis[val_analysis['y_true'] == 'bleaching']

print("--- False negative rate by Ocean ---")
for ocean in total_bleaching['Ocean_Name'].unique():
    total = len(total_bleaching[total_bleaching['Ocean_Name'] == ocean])
    fn = len(false_negatives[false_negatives['Ocean_Name'] == ocean])
    print(f"{ocean}: {fn}/{total} ({fn/total:.2f})")

print("\n--- False negative rate by Realm ---")
for realm in total_bleaching['Realm_Name'].unique():
    total = len(total_bleaching[total_bleaching['Realm_Name'] == realm])
    fn = len(false_negatives[false_negatives['Realm_Name'] == realm])
    print(f"{realm}: {fn}/{total} ({fn/total:.2f})")

--- False negative rate by Ocean ---
Indian: 0/11 (0.00)
Pacific: 236/319 (0.74)
Arabian Gulf: 2/4 (0.50)
Atlantic: 107/564 (0.19)
Red Sea: 4/6 (0.67)

--- False negative rate by Realm ---
Western Indo-Pacific: 6/19 (0.32)
Central Indo-Pacific: 204/277 (0.74)
Tropical Atlantic: 107/564 (0.19)
Temperate Northern Pacific: 12/19 (0.63)
Eastern Indo-Pacific: 2/6 (0.33)
Tropical Eastern Pacific: 6/7 (0.86)
Temperate Australasia: 12/12 (1.00)


In [33]:
# False positives: none events predicted as bleaching
false_positives = val_analysis[
    (val_analysis['y_true'] == 'none') & 
    (val_analysis['y_pred'] == 'bleaching')
]

print(f"Total false positives: {len(false_positives)}")
print(f"Total none events: {len(val_analysis[val_analysis['y_true'] == 'none'])}")
print(f"False alarm rate: {len(false_positives) / len(val_analysis[val_analysis['y_true'] == 'none']):.2f}")

print("\n--- False positives by Ocean ---")
for ocean in val_analysis['Ocean_Name'].unique():
    total = len(val_analysis[(val_analysis['y_true'] == 'none') & (val_analysis['Ocean_Name'] == ocean)])
    fp = len(false_positives[false_positives['Ocean_Name'] == ocean])
    if total > 0:
        print(f"{ocean}: {fp}/{total} ({fp/total:.2f})")

print("\n--- False positives by Realm ---")
for realm in val_analysis['Realm_Name'].unique():
    total = len(val_analysis[(val_analysis['y_true'] == 'none') & (val_analysis['Realm_Name'] == realm)])
    fp = len(false_positives[false_positives['Realm_Name'] == realm])
    if total > 0:
        print(f"{realm}: {fp}/{total} ({fp/total:.2f})")

Total false positives: 628
Total none events: 6327
False alarm rate: 0.10

--- False positives by Ocean ---
Indian: 1/433 (0.00)
Pacific: 100/3860 (0.03)
Arabian Gulf: 0/166 (0.00)
Atlantic: 527/1535 (0.34)
Red Sea: 0/333 (0.00)

--- False positives by Realm ---
Western Indo-Pacific: 1/936 (0.00)
Central Indo-Pacific: 76/3082 (0.02)
Tropical Atlantic: 527/1535 (0.34)
Temperate Northern Pacific: 5/153 (0.03)
Eastern Indo-Pacific: 17/404 (0.04)
Temperate Australasia: 0/211 (0.00)
Tropical Eastern Pacific: 2/6 (0.33)


### 6.1.1 False Positive Analysis

The model generates 628 false positives out of 6,327 none events (false alarm rate: 0.10).

**By Ocean:**
- Atlantic: 527/1535 (0.34): by far the highest false alarm rate
- Pacific: 100/3860 (0.03): very low
- Indian, Red Sea, Arabian Gulf: near zero

**By Realm:**
- Tropical Atlantic: 527/1535 (0.34): almost all Atlantic false alarms
- Tropical Eastern Pacific: 2/6 (0.33): high rate but tiny sample
- All Indo-Pacific realms: below 0.04

**Interpretation:**
The model is over-confident on the Atlantic and under-confident on the Indo-Pacific.
This is the mirror image of the false negative pattern:

- Atlantic: low miss rate (0.19) but high false alarm rate (0.34)
- Central Indo-Pacific: high miss rate (0.74) but near-zero false alarm rate (0.02)

The model has learned the Atlantic thermal signal well enough to over-trigger,
while it remains too conservative on the Pacific and Indo-Pacific regions
due to underrepresentation in the training set.

This asymmetry confirms that a single global model cannot perform equally well
across all ocean basins. Regional models are the recommended long-term solution.

### 6.2 Feature ablation: removing Date_Year

We test whether removing `Date_Year` from the feature set improves or worsens 
performance. The concern is that the model may learn "recent year = bleaching" 
as a shortcut rather than learning the underlying thermal signal.

In [34]:
# Remove Date_Year from feature set
num_cols_no_year = [col for col in num_cols if col != 'Date_Year']

# New preprocessor without Date_Year
preprocessor_no_year = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols_no_year),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
])

# Reprocess train, val, test
X_train_no_year = X_train_bin.drop(columns='Date_Year')
X_val_no_year = X_val_bin.drop(columns='Date_Year')
X_test_no_year = X_test_bin.drop(columns='Date_Year')

X_train_no_year_processed = preprocessor_no_year.fit_transform(X_train_no_year)
X_val_no_year_processed = preprocessor_no_year.transform(X_val_no_year)

print(f"Train processed: {X_train_no_year_processed.shape}")
print(f"Val processed:   {X_val_no_year_processed.shape}")

Train processed: (18121, 37)
Val processed:   (7231, 37)


In [35]:
# SMOTE on train without Date_Year
X_train_no_year_smote, y_train_no_year_smote = smote.fit_resample(
    X_train_no_year_processed, y_train_bin
)

# Random Forest without Date_Year
rf_no_year = RandomForestClassifier(
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)
rf_no_year.fit(X_train_no_year_smote, y_train_no_year_smote)

y_pred_no_year = rf_no_year.predict(X_val_no_year_processed)

print(classification_report(y_val_bin, y_pred_no_year))

              precision    recall  f1-score   support

   bleaching       0.47      0.62      0.53       904
        none       0.94      0.90      0.92      6327

    accuracy                           0.86      7231
   macro avg       0.71      0.76      0.73      7231
weighted avg       0.88      0.86      0.87      7231



In [ ]:
cm = confusion_matrix(y_val_bin, y_pred_no_year, labels=['none', 'bleaching'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['none', 'bleaching'])

disp.plot(cmap=sns.light_palette(color_primary, as_cmap=True))
plt.title('Random Forest Binary — No Date_Year — Confusion Matrix (Validation Set)')
plt.tight_layout()
plt.show()

### 6.2.1 Date_Year Ablation Results

Removing `Date_Year` from the feature set produces no meaningful change in performance:
- Bleaching recall: 0.61 → 0.62 (within noise)
- Macro F1: 0.73 → 0.73 (unchanged)
- False negatives: 349 → 340 (marginal improvement)

**Conclusion:** `Date_Year` does not introduce measurable temporal overfitting 
in this model. The thermal variables (SSTA, TSA_DHW, etc.) carry the predictive 
signal, not the year itself. `Date_Year` is retained in the final model.

## 7. Hyperparameter Tuning: GridSearchCV

We use GridSearchCV to find the optimal hyperparameters for the Random Forest binary model.
GridSearchCV tests all combinations of the specified parameter grid and selects the one 
that maximises the chosen scoring metric on the validation folds.

We optimise for `recall` on the `bleaching` class: consistent with our primary metric.

In [38]:
# Define recall scorer on the bleaching class specifically
# Default sklearn recall would score on 'none' (alphabetically last) — wrong for our goal
recall_bleaching = make_scorer(recall_score, pos_label='bleaching')

param_grid = {
    'n_estimators': [100, 200, 500],   # number of trees
    'max_depth': [10, 20, 30],          # maximum tree depth
    'min_samples_leaf': [1, 2, 5],      # minimum samples per leaf node
    'max_features': ['sqrt', 'log2']    # features considered at each split
}

# Base model — no class_weight, SMOTE already handles imbalance
rf_grid = RandomForestClassifier(random_state=42)

# GridSearchCV: tests all 54 combinations, 5-fold CV each = 270 fits
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring=recall_bleaching,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_bin_smote, y_train_bin_smote)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best recall on bleaching (CV): {grid_search.best_score_:.3f}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best parameters: {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 'n_estimators': 200}
Best recall on bleaching (CV): 0.972


In [39]:
# Train model with best parameters on SMOTE train, evaluate on real validation set
rf_tuned = RandomForestClassifier(
    max_depth=20,
    max_features='log2',
    min_samples_leaf=1,
    n_estimators=200,
    random_state=42
)

rf_tuned.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_tuned = rf_tuned.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_tuned))

              precision    recall  f1-score   support

   bleaching       0.53      0.54      0.53       904
        none       0.93      0.93      0.93      6327

    accuracy                           0.88      7231
   macro avg       0.73      0.73      0.73      7231
weighted avg       0.88      0.88      0.88      7231



### 7.1 GridSearchCV — Results

**Best parameters found:** `max_depth=20`, `max_features='log2'`, 
`min_samples_leaf=1`, `n_estimators=200`

**CV recall on SMOTE train: 0.972**: artificially high for two compounding reasons:
1. **SMOTE applied before CV:** SMOTE was applied to the full training set before 
   GridSearchCV split it into folds. Each validation fold contains synthetic samples 
   whose nearest neighbours were drawn from the training portion of that same fold — 
   the model effectively sees the answers during validation.
2. **Random KFold on temporal data:** `cv=5` uses random folds, which mix observations 
   from different years. A fold may train on 2005–2009 and validate on 1995–2000, 
   violating the temporal ordering of the data.

These two problems together make the CV recall unreliable as a selection criterion. 
This is why GridSearchCV was ultimately abandoned in favour of manual tuning evaluated 
directly on the real, held-out validation set (2010–2014).

**Performance on real validation set:**

| Model          | Recall (bleaching) | Precision (bleaching) | Macro F1 |
|----------------|--------------------|-----------------------|----------|
| RF baseline    | 0.61               | 0.47                  | 0.73     |
| RF tuned       | 0.54               | 0.53                  | 0.73     |

Tuning improved precision but reduced recall — the wrong trade-off for this project.
The culprit is `min_samples_leaf=1`: very deep trees that memorise the SMOTE 
training data but generalise poorly to the real validation distribution.

**Next step:** re-run GridSearchCV excluding `min_samples_leaf=1`, 
focusing on values that prevent overfitting (5, 10, 20).

In [ ]:
# Second GridSearch — excluding min_samples_leaf=1 which caused overfitting
# Higher min_samples_leaf values force more conservative trees
param_grid_v2 = {
    'n_estimators': [100, 200, 500],    # number of trees
    'max_depth': [10, 20, 30],           # maximum tree depth
    'min_samples_leaf': [5, 10, 20],     # minimum samples per leaf — higher = more conservative
    'max_features': ['sqrt', 'log2']     # features considered at each split
}

# Base model — no class_weight, SMOTE already handles imbalance
rf_grid2 = RandomForestClassifier(random_state=42)

# 54 combinations x 5 folds = 270 fits
grid_search_v2 = GridSearchCV(
    estimator=rf_grid2,
    param_grid=param_grid_v2,
    scoring=recall_bleaching,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search_v2.fit(X_train_bin_smote, y_train_bin_smote)

print(f"Best parameters: {grid_search_v2.best_params_}")
print(f"Best recall on bleaching (CV): {grid_search_v2.best_score_:.3f}")

In [41]:
rf_tuned_v2 = RandomForestClassifier(
    max_depth=20,
    max_features='sqrt',
    min_samples_leaf=5,
    n_estimators=500,
    random_state=42
)

rf_tuned_v2.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_tuned_v2 = rf_tuned_v2.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_tuned_v2))

              precision    recall  f1-score   support

   bleaching       0.52      0.55      0.54       904
        none       0.94      0.93      0.93      6327

    accuracy                           0.88      7231
   macro avg       0.73      0.74      0.73      7231
weighted avg       0.88      0.88      0.88      7231



### 7.2 GridSearchCV v2 Results

**Best parameters found:** `max_depth=20`, `max_features='sqrt'`, 
`min_samples_leaf=5`, `n_estimators=500`

**CV recall on SMOTE train: 0.959**, still artificially high for the same reason as v1.

**Performance on real validation set:**

| Model            | Recall (bleaching) | Precision (bleaching) | Macro F1 |
|------------------|--------------------|-----------------------|----------|
| RF baseline      | 0.61               | 0.47                  | 0.73     |
| RF tuned v1      | 0.54               | 0.53                  | 0.73     |
| RF tuned v2      | 0.55               | 0.52                  | 0.73     |

Excluding `min_samples_leaf=1` produced a marginal improvement over v1 (0.54 → 0.55) 
but recall remains below the baseline.

**Root cause:** GridSearchCV optimises on the balanced SMOTE training set. 
Parameters that maximise recall on synthetic 50/50 data do not generalise 
to the real validation distribution (82/18). Deeper trees (max_depth=20) 
learn the SMOTE data well but overfit relative to the baseline (max_depth=10).

**Next step:** manual tuning, vary one parameter at a time, 
evaluate directly on the real validation set.

In [42]:
# Manual tuning experiment 1
# Hypothesis: lower max_depth reduces overfitting and improves recall on real data
# Baseline was max_depth=10, GridSearch found max_depth=20 which hurt recall
# Testing max_depth=8 to see if shallower trees generalise better

rf_manual_1 = RandomForestClassifier(
    max_depth=8,           # shallower than baseline (10) 
    min_samples_leaf=5,    # same as baseline
    n_estimators=200,      # more trees than baseline (100) for stability
    max_features='sqrt',   # standard choice for classification
    random_state=42
)

rf_manual_1.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_1 = rf_manual_1.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_1))

              precision    recall  f1-score   support

   bleaching       0.46      0.63      0.53       904
        none       0.94      0.89      0.92      6327

    accuracy                           0.86      7231
   macro avg       0.70      0.76      0.72      7231
weighted avg       0.88      0.86      0.87      7231



In [43]:
# Manual tuning experiment 2
# Hypothesis: even shallower trees may generalise better
# Testing max_depth=6

rf_manual_2 = RandomForestClassifier(
    max_depth=6,           # shallower than experiment 1 (8)
    min_samples_leaf=5,    # same as baseline
    n_estimators=200,      # same as experiment 1
    max_features='sqrt',   # same as experiment 1
    random_state=42
)

rf_manual_2.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_2 = rf_manual_2.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_2))

              precision    recall  f1-score   support

   bleaching       0.44      0.64      0.52       904
        none       0.95      0.88      0.91      6327

    accuracy                           0.85      7231
   macro avg       0.69      0.76      0.72      7231
weighted avg       0.88      0.85      0.86      7231



In [44]:
# Manual tuning experiment 3
# Continuing to reduce max_depth to find the sweet spot
# Testing max_depth=5

rf_manual_3 = RandomForestClassifier(
    max_depth=5,           # shallower than experiment 2 (6)
    min_samples_leaf=5,    # same as baseline
    n_estimators=200,      # same as experiments 1 and 2
    max_features='sqrt',   # same as experiments 1 and 2
    random_state=42
)

rf_manual_3.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_3 = rf_manual_3.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_3))

              precision    recall  f1-score   support

   bleaching       0.44      0.64      0.52       904
        none       0.94      0.88      0.91      6327

    accuracy                           0.85      7231
   macro avg       0.69      0.76      0.72      7231
weighted avg       0.88      0.85      0.86      7231



In [45]:
# Manual tuning experiment 4
# max_depth=6 seems to be the sweet spot
# Now testing min_samples_leaf=3 to see if slightly smaller leaves help recall

rf_manual_4 = RandomForestClassifier(
    max_depth=6,           # sweet spot found in experiment 2
    min_samples_leaf=3,    # slightly smaller than baseline (5)
    n_estimators=200,
    max_features='sqrt',
    random_state=42
)

rf_manual_4.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_4 = rf_manual_4.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_4))

              precision    recall  f1-score   support

   bleaching       0.44      0.64      0.52       904
        none       0.94      0.88      0.91      6327

    accuracy                           0.85      7231
   macro avg       0.69      0.76      0.72      7231
weighted avg       0.88      0.85      0.86      7231



In [ ]:
# Manual tuning experiment 5
# Testing n_estimators=300 as first intermediate step between 200 and 500

rf_manual_5 = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=5,
    n_estimators=300,
    max_features='sqrt',
    random_state=42
)

rf_manual_5.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_5 = rf_manual_5.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_5))

              precision    recall  f1-score   support

   bleaching       0.44      0.63      0.52       904
        none       0.94      0.89      0.91      6327

    accuracy                           0.86      7231
   macro avg       0.69      0.76      0.72      7231
weighted avg       0.88      0.86      0.87      7231



In [49]:
# Manual tuning experiment 6
# Testing n_estimators=500

rf_manual_6 = RandomForestClassifier(
    max_depth=6,
    min_samples_leaf=5,
    n_estimators=500,
    max_features='sqrt',
    random_state=42
)

rf_manual_6.fit(X_train_bin_smote, y_train_bin_smote)
y_pred_manual_6 = rf_manual_6.predict(X_val_bin_processed)

print(classification_report(y_val_bin, y_pred_manual_6))

              precision    recall  f1-score   support

   bleaching       0.45      0.63      0.52       904
        none       0.94      0.89      0.92      6327

    accuracy                           0.86      7231
   macro avg       0.70      0.76      0.72      7231
weighted avg       0.88      0.86      0.87      7231



### 7.3 Manual Tuning — Results

GridSearchCV optimised on SMOTE data consistently underperformed the baseline 
on the real validation set. Manual tuning, varying one parameter at a time 
and evaluating directly on the real validation distribution, proved more effective.

**Key finding:** shallower trees generalise better on this dataset.
Reducing `max_depth` from 10 (baseline) to 6 improved bleaching recall from 0.61 to 0.64.

| Experiment | max_depth | min_samples_leaf | n_estimators | Recall (bleaching) |
|------------|-----------|------------------|--------------|--------------------|
| Baseline   | 10        | 5                | 100          | 0.61               |
| Manual 1   | 8         | 5                | 200          | 0.63               |
| Manual 2   | 6         | 5                | 200          | 0.64               |
| Manual 3   | 5         | 5                | 200          | 0.64               |
| Manual 4   | 6         | 3                | 200          | 0.64               |
| Manual 5   | 6         | 5                | 300          | 0.63               |
| Manual 6   | 6         | 5                | 500          | 0.63               |

**Conclusion:** `max_depth=6`, `min_samples_leaf=5`, `n_estimators=200`, 
`max_features='sqrt'` is the final model configuration.
Recall plateau at 0.64. This is the structural ceiling of a global model 
on this dataset, as confirmed by the error analysis in section 6.
Further improvement requires regional models or an extended dataset (post-2020).

## 8. Model Interpretability: SHAP

SHAP (SHapley Additive exPlanations) quantifies the contribution of each feature 
to individual model predictions, and explains each prediction individually.

In this project, interpretability is as important as performance: understanding 
which environmental variables drive bleaching predictions helps identify where 
to focus conservation efforts and which regions are more at risk.

We expect thermal stress variables (SSTA, TSA_DHW, DHW) to dominate, 
consistent with the coral bleaching literature. Local physical variables 
(turbidity, windspeed, depth) are expected to play a secondary role.

In [ ]:
# Creating the SHAP explainer using the final tuned model
explainer = shap.TreeExplainer(rf_manual_2)

# SHAP values on real validation data, not SMOTE synthetic samples
# shap_values is a list of two arrays: [class_0 (bleaching), class_1 (none)]
shap_sample = X_val_bin_processed[:500]
shap_values = explainer.shap_values(shap_sample)

In [62]:
# Checking the names of all features
preprocessor_bin.get_feature_names_out()

array(['num__Latitude_Degrees', 'num__Longitude_Degrees',
       'num__Distance_to_Shore', 'num__Turbidity',
       'num__Cyclone_Frequency', 'num__Date_Month', 'num__Date_Year',
       'num__Depth_m', 'num__ClimSST', 'num__Temperature_Kelvin',
       'num__Temperature_Mean', 'num__Temperature_Maximum',
       'num__Windspeed', 'num__SSTA', 'num__SSTA_DHW', 'num__TSA',
       'num__TSA_Maximum', 'num__TSA_Mean', 'num__TSA_Frequency',
       'num__TSA_DHW', 'num__TSA_DHWMax', 'num__TSA_DHWMean',
       'cat__Ocean_Name_Atlantic', 'cat__Ocean_Name_Indian',
       'cat__Ocean_Name_Pacific', 'cat__Ocean_Name_Red Sea',
       'cat__Realm_Name_Eastern Indo-Pacific',
       'cat__Realm_Name_Temperate Australasia',
       'cat__Realm_Name_Temperate Northern Atlantic',
       'cat__Realm_Name_Temperate Northern Pacific',
       'cat__Realm_Name_Tropical Atlantic',
       'cat__Realm_Name_Tropical Eastern Pacific',
       'cat__Realm_Name_Western Indo-Pacific',
       'cat__Substrate_Name_Hard C

In [ ]:
shap.summary_plot(
    shap_values[:, :, 0],
    shap_sample,
    feature_names=preprocessor_bin.get_feature_names_out(),
    plot_type="bar",
    max_display=20,
    color=color_primary
)
plt.show()

### 8.1 SHAP Results and Interpretation

**Top features by mean absolute SHAP value:**

1. `Substrate_Name_Not_Recorded` (≈0.13) — dominates the plot
2. `Substrate_Name_Hard Coral` and `Substrate_Name_Nutrient Indicator Algae` (≈0.04)
3. Geographic variables: `Longitude`, `Ocean_Atlantic`, `Realm_Tropical_Atlantic` (≈0.04)
4. `TSA_DHW` — only 7th place (≈0.03)

**Three key observations:**

**1. `Substrate_Name_Not_Recorded` is a red flag.**
This is not an ecological variable: it is the absence of information. 
The model uses it as a proxy because "Not_Recorded" is systematically associated 
with certain studies, regions, or historical periods in which substrate was not recorded.
This is a dataset artefact, not a biological signal.

**2. Geographic variables dominate over thermal variables.**
The model has learned *where* bleaching happens rather than *why* it happens.
This confirms the geographic bias identified in §6: the model over-predicts 
bleaching in the Tropical Atlantic and under-predicts bleaching in the Indo-Pacific 
and Pacific.

**3. Thermal stress variables are underrepresented.**
`TSA_DHW`, the most established thermal bleaching indicator in the literature, 
ranks only 7th. This does not mean thermal stress is ecologically unimportant — 
it means the model has not fully learned the thermal signal. Geographic proxies 
carry more discriminating power within the training distribution, 
suppressing the thermal variables in importance rankings.

**Conclusion:**
SHAP confirms that the global model has structural limitations: it has learned 
to discriminate bleaching by geography and substrate recording patterns rather 
than thermal physics. This is consistent with the recall plateau at 0.64 observed 
in §7.3, and motivates the regional model approach explored in §9.

## 9. Regional Model — Atlantic Ocean

In [64]:
# Atlantic observations in the full dataset
print(df['Ocean_Name'].value_counts())
print(f"\nAtlantic percentage: {df[df['Ocean_Name'] == 'Atlantic'].shape[0] / df.shape[0]:.2%}")

Ocean_Name
Pacific         16264
Atlantic        13087
Indian           1982
Red Sea          1037
Arabian Gulf      341
Name: count, dtype: int64

Atlantic percentage: 40.01%


In [65]:
# Atlantic observations by year
atlantic_by_year = df[df['Ocean_Name'] == 'Atlantic'].groupby('Date_Year').size()
print(atlantic_by_year)

Date_Year
1987      16
1988       1
1990       1
1991       2
1992       3
1993       4
1994       3
1995       3
1997       2
1998      84
1999     713
2000     514
2001     642
2002     387
2003     702
2004     490
2005    2792
2006    1513
2007     619
2008     630
2009     621
2010     472
2011     523
2012     444
2013     291
2014     369
2015     460
2016     324
2017     221
2018     167
2019      74
dtype: int64


In [66]:
# Pacific observations by year
pacific_by_year = df[df['Ocean_Name'] == 'Pacific'].groupby('Date_Year').size()
print(pacific_by_year)

Date_Year
1983       3
1991       1
1992       2
1993       2
1994       8
1996      10
1997       4
1998      67
1999       4
2000      32
2001      47
2002     175
2003     819
2004    1032
2005     814
2006    1134
2007     755
2008    1060
2009    1028
2010     893
2011     616
2012     724
2013    1070
2014     876
2015     942
2016    1182
2017    1176
2018     912
2019     876
dtype: int64


In [67]:
# Observations per ocean per split period
for ocean in ['Atlantic', 'Pacific']:
    print(f"\n--- {ocean} ---")
    train_count = df[(df['Ocean_Name'] == ocean) & (df['Date_Year'] <= 2009)].shape[0]
    val_count = df[(df['Ocean_Name'] == ocean) & (df['Date_Year'] >= 2010) & (df['Date_Year'] <= 2014)].shape[0]
    test_count = df[(df['Ocean_Name'] == ocean) & (df['Date_Year'] >= 2015)].shape[0]
    print(f"Train (1980-2009): {train_count}")
    print(f"Val   (2010-2014): {val_count}")
    print(f"Test  (2015-2019): {test_count}")


--- Atlantic ---
Train (1980-2009): 9742
Val   (2010-2014): 2099
Test  (2015-2019): 1246

--- Pacific ---
Train (1980-2009): 6997
Val   (2010-2014): 4179
Test  (2015-2019): 5088


### 9.1 Why a Regional Atlantic Model Makes Sense

The temporal split reveals a structural asymmetry between the Atlantic and Pacific:

- Atlantic train (1980-2009): 9,742 observations
- Pacific train (1980-2009): 6,997 observations

The raw numbers look comparable, but the events are not. The Atlantic's two major 
bleaching episodes (1998 and 2005) both fall within the training window. The Pacific's 
most devastating events, 2016, 2020, 2022, and 2024, fall entirely outside the training 
set, or are barely captured in the test set.

This is not a sample size problem. It is a temporal distribution problem. The model 
has never seen a massive Pacific bleaching event during training, so it cannot learn 
the Pacific thermal signal. With data extending to 2024, at least four major Pacific 
events would be available for training, and the model would have real signal to learn from.

The global model's Atlantic bias is therefore structural: it reflects the documented 
history of coral monitoring pre-2010, not a geographically balanced distribution 
of bleaching events. A regional Atlantic model is the honest and scientifically 
defensible choice for this dataset and this timeline.